# 🎨 Blender GPU Render Worker — Jalur Hybrid (Colab)

Worker render **transient** untuk arsitektur hybrid:

```
VPS dedicated (otak/host, Selkies GUI)
  └─ Claude + cli-anything-blender  →  generate _render_script.py / scene.json
        ⇣  (Google Drive)
  Colab GPU (notebook ini)  →  blender --background --python  →  PNG/EXR
        ⇡  (Drive)
```

**Catatan ToS:** notebook ini untuk render **interaktif/burst**, bukan server 24/7 atau
job otomatis tanpa henti — itu melanggar ToS Colab. Host persisten tetap di VPS.

**Wajib:** set runtime **GPU** (Runtime ▸ Change runtime type ▸ T4 GPU) sebelum mulai.

In [ ]:
#@title 0 · Cek runtime GPU
!nvidia-smi -L || echo "⚠️  Tidak ada GPU — notebook tetap jalan tapi render via CPU. Set Runtime ▸ GPU."


In [ ]:
#@title ⚙️ Konfigurasi
BLENDER_SERIES = "4.2"  #@param {type:"string"}
# 4.2 = LTS; enum 'BLENDER_EEVEE_NEXT' yang di-generate harness VALID di sini,
# jadi script EEVEE dari VPS (Blender 5.x) jalan tanpa patch.
DRIVE_ROOT = "/content/drive/MyDrive/blender-hybrid"  #@param {type:"string"}
import os
os.environ["BLENDER_SERIES"] = BLENDER_SERIES  # agar sel %%bash ikut memakainya
IN_DIR    = f"{DRIVE_ROOT}/in"
OUT_DRIVE = f"{DRIVE_ROOT}/out"
print("series :", BLENDER_SERIES)
print("in     :", IN_DIR)
print("out    :", OUT_DRIVE)


## 1 · Pasang Blender LTS + dependency headless

In [ ]:
#@title 1 · Install Blender + cli-anything-blender
%%bash
set -e
echo "**** dependency headless ****"
# Repo pihak-ketiga bawaan Colab (r2u / nvidia-cuda) kadang gagal sync dan bikin
# `apt-get update` exit!=0. Update HANYA dari sources.list utama + jadikan non-fatal.
apt-get update -o Dir::Etc::sourceparts=/dev/null -o APT::Get::List-Cleanup=0 2>/dev/null \
  || apt-get update 2>/dev/null || true
apt-get install -y --no-install-recommends \
  libxrender1 libxi6 libxxf86vm1 libxfixes3 libxkbcommon0 libgl1 libsm6 2>/dev/null \
  || echo "WARN: sebagian lib gagal/sudah ada -- lanjut."

BL_SERIES=${BLENDER_SERIES:-4.2}
BASE="https://download.blender.org/release/Blender${BL_SERIES}"
FILE=$(curl -s "$BASE/" | grep -oP "blender-${BL_SERIES}\\.[0-9]+-linux-x64\\.tar\\.xz" | sort -V | tail -1)
echo "**** download $FILE ****"
curl -fL -o /tmp/blender.tar.xz "$BASE/$FILE"
mkdir -p /opt/blender
tar xf /tmp/blender.tar.xz -C /opt/blender --strip-components=1
ln -sf /opt/blender/blender /usr/local/bin/blender
rm -f /tmp/blender.tar.xz
blender --version | head -1

echo "**** harness (opsional, utk membangun scene di Colab) ****"
pip -q install cli-anything-blender
cli-anything-blender --help >/dev/null && echo "cli-anything-blender OK"


## 2 · Mount Google Drive (jembatan VPS ⇄ Colab)

Di VPS, salin artifact ke folder Drive yang sama, mis.:
`rclone copy /config/_render_script.py gdrive:blender-hybrid/in/`

In [ ]:
#@title 2 · Mount Drive + siapkan folder
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs(IN_DIR, exist_ok=True)
os.makedirs(OUT_DRIVE, exist_ok=True)
print("in :", IN_DIR, "->", os.listdir(IN_DIR))
print("out:", OUT_DRIVE)


## 3 · GPU prelude

Script ini dijalankan SEBELUM `_render_script.py` dalam proses Blender yang sama.
Ia mengaktifkan device GPU (OPTIX→CUDA→HIP→ONEAPI) dan memasang handler `render_pre`
yang memaksa **Cycles ke GPU** + mengarahkan output ke folder Colab — tanpa perlu
mengedit script hasil harness.

In [ ]:
#@title 3 · Tulis gpu_prelude.py
%%writefile /content/gpu_prelude.py
import bpy, os
from bpy.app.handlers import persistent

OUT_PREFIX = os.environ.get("BL_OUT_PREFIX", "/content/out/render_")
USE_GPU = os.environ.get("BL_USE_GPU", "1") == "1"

def _enable_gpu():
    prefs = bpy.context.preferences.addons["cycles"].preferences
    chosen = None
    for backend in ("OPTIX", "CUDA", "HIP", "ONEAPI"):
        try:
            prefs.compute_device_type = backend
            chosen = backend
            break
        except TypeError:
            continue
    if chosen is None:
        print("[prelude] tidak ada backend GPU -> CPU")
        return False
    prefs.get_devices()
    n = 0
    for d in prefs.devices:
        d.use = (d.type != "CPU")
        if d.use:
            n += 1
            print(f"[prelude] GPU: {d.name} ({d.type})")
    print(f"[prelude] backend={chosen} gpu_devices={n}")
    return n > 0

gpu_ok = _enable_gpu() if USE_GPU else False

@persistent
def _on_render(scene, *args):
    try:
        if scene.render.engine == "CYCLES":
            scene.cycles.device = "GPU" if gpu_ok else "CPU"
    except Exception as e:
        print("[prelude] warn device:", e)
    os.makedirs(os.path.dirname(OUT_PREFIX), exist_ok=True)
    scene.render.filepath = OUT_PREFIX
    print(f"[prelude] engine={scene.render.engine} "
          f"device={getattr(scene.cycles,'device','-')} out={OUT_PREFIX}")

bpy.app.handlers.render_pre.append(_on_render)
print("[prelude] siap (gpu_ok=%s)" % gpu_ok)


## 4 · Fungsi render & helper

In [ ]:
#@title 4 · Helper render
import os, subprocess, glob, shutil, time
from IPython.display import Image, display

def render_script_on_gpu(script_path, job_name="job", use_gpu=True):
    """Render sebuah _render_script.py hasil harness, paksa Cycles ke GPU."""
    out_dir = "/content/out"
    shutil.rmtree(out_dir, ignore_errors=True)
    os.makedirs(out_dir, exist_ok=True)
    env = dict(os.environ,
               BL_OUT_PREFIX=f"{out_dir}/{job_name}_",
               BL_USE_GPU="1" if use_gpu else "0")
    t0 = time.time()
    p = subprocess.run(
        ["blender", "--background",
         "--python", "/content/gpu_prelude.py",
         "--python", script_path],
        env=env, capture_output=True, text=True)
    dt = time.time() - t0
    print(p.stdout[-3000:])
    if p.returncode != 0:
        print("STDERR:", p.stderr[-2000:])
        raise RuntimeError(f"Blender exit {p.returncode}")
    outs = sorted(glob.glob(f"{out_dir}/*"))
    print(f"\n✅ render {dt:.1f}s -> {outs}")
    return outs

def publish_to_drive(files, drive_out):
    os.makedirs(drive_out, exist_ok=True)
    for f in files:
        shutil.copy(f, drive_out)
    print("💾 tersimpan di", drive_out, "->", os.listdir(drive_out))

def show(files):
    for f in files:
        if f.lower().endswith((".png", ".jpg", ".jpeg")):
            display(Image(f))

def build_scene_colab(project, commands, engine="CYCLES", samples=64,
                      rx=640, ry=480, out="/content/out.png"):
    """Bangun scene via harness (REPL+save) lalu generate _render_script.py."""
    subprocess.run(["cli-anything-blender", "--json", "scene", "new", "-o", project],
                   check=True, capture_output=True, text=True)
    repl = "\n".join(commands + [
        f"render settings --engine {engine} --samples {samples} "
        f"-rx {rx} -ry {ry} --output-path {out}",
        "scene save", "exit", ""])
    subprocess.run(["cli-anything-blender", "--project", project],
                   input=repl, text=True, capture_output=True)
    subprocess.run(["cli-anything-blender", "--json", "--project", project,
                    "render", "execute", out, "--overwrite"],
                   check=True, capture_output=True, text=True)
    return os.path.join(os.path.dirname(out) or ".", "_render_script.py")

print("helper siap")


## 5 · Mode A — render `_render_script.py` dari VPS

Letakkan `_render_script.py` (hasil `cli-anything-blender render execute` di VPS) di
`blender-hybrid/in/` pada Drive, lalu jalankan sel ini.

In [ ]:
#@title 5 · Render dari VPS (Mode A)
script = f"{IN_DIR}/_render_script.py"
if not os.path.exists(script):
    print("⚠️  Belum ada", script, "— upload dulu dari VPS. Lewati ke Mode B utk smoke test.")
else:
    outs = render_script_on_gpu(script, job_name="fromVPS", use_gpu=True)
    publish_to_drive(outs, OUT_DRIVE)
    show(outs)


## 6 · Mode B — smoke test (bangun scene di Colab + render GPU)

Membuktikan seluruh jalur tanpa perlu VPS: bangun Suzanne, render Cycles di GPU.

In [ ]:
#@title 6 · Smoke test (Mode B)
script = build_scene_colab(
    project="/content/scene.json",
    commands=["object add monkey", "camera add", "light add sun"],
    engine="CYCLES", samples=64, rx=640, ry=480, out="/content/out.png")
print("script:", script)
outs = render_script_on_gpu(script, job_name="suzanne", use_gpu=True)
publish_to_drive(outs, OUT_DRIVE)
show(outs)


## 7 · Catatan

- **Engine:** `CYCLES` → otomatis dipaksa ke GPU (lihat prelude). `WORKBENCH`/`EEVEE`
  tetap jalan; EEVEE valid karena Blender 4.2 LTS menerima `BLENDER_EEVEE_NEXT`.
- **Output:** prelude mengarahkan semua output ke `/content/out/` lalu disalin ke Drive;
  path asli `/config/...` dari VPS diabaikan dengan aman.
- **Verifikasi GPU:** cari baris `[prelude] backend=OPTIX gpu_devices=1` di output.
- **Animasi:** untuk sekuens, render banyak frame menumpuk cepat — pantau kuota Drive.
- **Sesi ephemeral:** unduh hasil penting; runtime bisa di-reset kapan saja.